In [1]:
!pip install -q "smolagents[openai]" pytz requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.9 MB/s eta 0:00:00


In [2]:
import os
import datetime
import pytz
import urllib.parse
import requests
from getpass import getpass

from smolagents import CodeAgent, OpenAIModel, FinalAnswerTool, tool

os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

model = OpenAIModel(
    model_id="openrouter/free",
    api_base="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens=2048,
    temperature=0.5,
)

OpenRouter API key: ··········


In [3]:
@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """A tool that fetches the current local time in a specified timezone.
    Args:
        timezone: A valid timezone name, for example 'Asia/Kolkata' or 'America/New_York'.
    """
    try:
        tz = pytz.timezone(timezone)
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {str(e)}"


In [4]:
@tool
def image_generation_tool(prompt: str) -> str:
    """Generate an image from a text prompt using the free Pollinations image API.
    Args:
        prompt: The image description to generate.
    """
    encoded_prompt = urllib.parse.quote(prompt)
    image_url = f"https://image.pollinations.ai/prompt/{encoded_prompt}"

    response = requests.get(image_url, timeout=60)
    if response.status_code != 200:
        return f"Image generation failed with status code {response.status_code}"

    return image_url


In [6]:
final_answer = FinalAnswerTool()


In [7]:
agent = CodeAgent(
    model=model,
    tools=[
        final_answer,
        get_current_time_in_timezone,
        image_generation_tool,
    ],
    max_steps=3,
)

agent.run("Tell me the current time in Asia/Kolkata.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me the current time in Asia/Kolkata.                                                                       │
│                                                                                                                 │
╰─ OpenAIModel - openrouter/free ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  current_time = get_current_time_in_timezone(timezone="Asia/Kolkata")                                             
  final_answer(current_time)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The current local time in Asia/Kolkata is: 2026-05-24 13:18:01

[Step 1: Duration 10.25 seconds| Input tokens: 2,178 | Output tokens: 375]

'The current local time in Asia/Kolkata is: 2026-05-24 13:18:01'

In [8]:
agent = CodeAgent(
    model=model,
    tools=[
        final_answer,
        get_current_time_in_timezone,
        image_generation_tool,
    ],
    max_steps=3,
)

agent.run(" generate an image of a futuristic cat.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ generate an image of a futuristic cat.                                                                          │
│                                                                                                                 │
╰─ OpenAIModel - openrouter/free ─────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  prompt = "A futuristic cat with sleek metallic fur, glowing neon-blue eyes, and cybernetic enhancements,         
  standing in a high-tech cityscape with holographic elements and neon lights."                                    
  image = image_generation_tool(prompt=prompt)                                                                     
  final_answer(image)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 
https://image.pollinations.ai/prompt/A%20futuristic%20cat%20with%20sleek%20metallic%20fur%2C%20glowing%20neon-blue%
20eyes%2C%20and%20cybernetic%20enhancements%2C%20standing%20in%20a%20high-tech%20cityscape%20with%20holographic%20e
lements%20and%20neon%20lights.

[Step 1: Duration 22.95 seconds| Input tokens: 2,176 | Output tokens: 617]

'https://image.pollinations.ai/prompt/A%20futuristic%20cat%20with%20sleek%20metallic%20fur%2C%20glowing%20neon-blue%20eyes%2C%20and%20cybernetic%20enhancements%2C%20standing%20in%20a%20high-tech%20cityscape%20with%20holographic%20elements%20and%20neon%20lights.'